In [ ]:
" Part 1: Reverse-Engineering Non-Stationarity "
" MW VS EtE configuration: ~ 10 seconds | MW VS UCB configuration: ~ 1 minute "

import numpy as np
import matplotlib.pyplot as plt

# PARAMETERS
K = 2
T = 10000    # Change to 20000 for extra credit
epochs = 10  # Change to 50 for extra credit
N = int(round(T**(2/3)))
epsilon = (K * np.log(K) / T)**(1/3)
eta = epsilon
window = 200 # Change to 100 for extra credit

# MW VS EtE
# Arm 0 Configuration
p0_L, p0_H = 0, 1
T0_L, T0_H = 2*N, T - 2*N
start0 = "Low"

# Arm 1 Configuration
p1_L, p1_H = 0, 1
T1_L, T1_H = T - 2*N, 2*N
start1 = "High"

"""
# MW VS UCB
# Arm 0 Configuration
p0_L, p0_H = 0, 1
T0_L, T0_H = 0.575*T, 0.425*T
start0 = "High"

# Arm 1 Configuration
p1_L, p1_H = 0.45, 0.9
T1_L, T1_H = 0.575*T, 0.425*T
start1 = "High"
"""

# Offsets
offset0 = 0 if start0 == "Low" else T0_L
offset1 = 0 if start1 == "Low" else T1_L

# Get current probability and state (High/Low)
def get_prob_and_state(arm, t):

    if arm == 0:
        pos = (t + offset0) % (T0_L + T0_H)
        if pos < T0_L:
            return p0_L
        else:
            return p0_H

    else:
        pos = (t + offset1) % (T1_L + T1_H)
        if pos < T1_L:
            return p1_L
        else:
            return p1_H

# Compute optimal expected cumulative reward
def get_optimal_expected_cumulative_reward(T):

    optimal = np.zeros(T)
    total = 0.0

    for t in range(T):
        p0 = get_prob_and_state(0, t)
        p1 = get_prob_and_state(1, t)

        total += max(p0, p1)
        optimal[t] = total

    return optimal

# Rolling mean function
def rolling_mean(x, window):
    return np.convolve(x, np.ones(window) / window, mode='valid')

# Arrays to store results over epochs
all_cum_reward_EtE = np.zeros((epochs, T))
all_cum_reward_UCB = np.zeros((epochs, T))
all_cum_reward_MW = np.zeros((epochs, T))

all_actions_EtE = np.zeros((epochs, T), dtype=int)
all_actions_UCB = np.zeros((epochs, T), dtype=int)
all_actions_MW  = np.zeros((epochs, T), dtype=int)

base_seed = 63

# MONTE CARLO
for ep in range(epochs):

    np.random.seed(base_seed + ep)

    # ======================
    # 1. Explore-then-Exploit
    # ======================
    rewards = np.zeros(T)
    arm0_rewards = []
    arm1_rewards = []

    # Exploration
    for t in range(N * K):

        arm = t % K
        all_actions_EtE[ep, t] = arm
        p = get_prob_and_state(arm, t)
        r = np.random.binomial(1, p)
        rewards[t] = r

        if arm == 0:
            arm0_rewards.append(r)
        else:
            arm1_rewards.append(r)

    mean0 = np.mean(arm0_rewards)
    mean1 = np.mean(arm1_rewards)

    if mean0 > mean1:
        best_arm = 0
    elif mean1 > mean0:
        best_arm = 1
    else:
        best_arm = np.random.choice([0, 1])

    # Exploitation
    for t in range(N * K, T):
        all_actions_EtE[ep, t] = best_arm
        p = get_prob_and_state(best_arm, t)
        r = np.random.binomial(1, p)
        rewards[t] = r

    all_cum_reward_EtE[ep] = np.cumsum(rewards)

    # ======================
    # 2. UCB
    # ======================
    rewards = np.zeros(T)
    Npull = np.zeros(K)
    sum_rewards = np.zeros(K)

    for t in range(T):
        # Select arm
        if t < K:
            arm = t
        else:
            exploit = sum_rewards / Npull
            explore = np.sqrt((2*np.log(T)) / Npull)
            ucb = exploit + explore
            arm = np.argmax(ucb)

        all_actions_UCB[ep, t] = arm
        p = get_prob_and_state(arm, t)
        r = np.random.binomial(1, p)

        rewards[t] = r

        Npull[arm] += 1
        sum_rewards[arm] += r

    all_cum_reward_UCB[ep] = np.cumsum(rewards)

    # ======================
    # 3. MW
    # ======================
    rewards = np.zeros(T)
    w = np.ones(K)

    for t in range(T):

        # Probability distribution from weights
        p = w / np.sum(w)

        # Adding exploration
        q = (1 - epsilon)*p + epsilon/K

        # Select arm randomly according to MW probabilities
        arm = np.random.choice(K, p=q)
        all_actions_MW[ep, t] = arm

        # Observe reward
        prob = get_prob_and_state(arm, t)
        r = np.random.binomial(1, prob)
        rewards[t] = r

        # Estimated loss vector
        loss_hat = np.zeros(K)
        loss_hat[arm] = (1 - r) / q[arm]

        # MW update
        w = w * ((1 - eta) ** loss_hat)

    all_cum_reward_MW[ep] = np.cumsum(rewards)

# AVERAGE RESULTS
mean_EtE = np.mean(all_cum_reward_EtE, axis=0)
mean_UCB = np.mean(all_cum_reward_UCB, axis=0)
mean_MW = np.mean(all_cum_reward_MW, axis=0)

# STANDARD DEVIATIONS (for the final round across all epochs)
std_EtE = np.std(all_cum_reward_EtE[:, -1])
std_UCB = np.std(all_cum_reward_UCB[:, -1])
std_MW = np.std(all_cum_reward_MW[:, -1])

# Optimal expected cumulative reward
optimal = get_optimal_expected_cumulative_reward(T)

# Rolling selection frequency of arms
rolling_arm0_EtE = np.zeros((epochs, T - window + 1))
rolling_arm0_UCB = np.zeros((epochs, T - window + 1))
rolling_arm0_MW  = np.zeros((epochs, T - window + 1))

rolling_arm1_EtE = np.zeros((epochs, T - window + 1))
rolling_arm1_UCB = np.zeros((epochs, T - window + 1))
rolling_arm1_MW  = np.zeros((epochs, T - window + 1))

for ep in range(epochs):
    # Arm 0
    pick0_EtE = (all_actions_EtE[ep] == 0).astype(float)
    pick0_UCB = (all_actions_UCB[ep] == 0).astype(float)
    pick0_MW  = (all_actions_MW[ep] == 0).astype(float)

    rolling_arm0_EtE[ep] = rolling_mean(pick0_EtE, window)
    rolling_arm0_UCB[ep] = rolling_mean(pick0_UCB, window)
    rolling_arm0_MW[ep]  = rolling_mean(pick0_MW, window)

    # Arm 1
    pick1_EtE = (all_actions_EtE[ep] == 1).astype(float)
    pick1_UCB = (all_actions_UCB[ep] == 1).astype(float)
    pick1_MW  = (all_actions_MW[ep] == 1).astype(float)

    rolling_arm1_EtE[ep] = rolling_mean(pick1_EtE, window)
    rolling_arm1_UCB[ep] = rolling_mean(pick1_UCB, window)
    rolling_arm1_MW[ep]  = rolling_mean(pick1_MW, window)

mean_rolling_arm0_EtE = np.mean(rolling_arm0_EtE, axis=0)
mean_rolling_arm0_UCB = np.mean(rolling_arm0_UCB, axis=0)
mean_rolling_arm0_MW  = np.mean(rolling_arm0_MW, axis=0)

mean_rolling_arm1_EtE = np.mean(rolling_arm1_EtE, axis=0)
mean_rolling_arm1_UCB = np.mean(rolling_arm1_UCB, axis=0)
mean_rolling_arm1_MW  = np.mean(rolling_arm1_MW, axis=0)

x_roll = np.arange(window, T + 1)

# PRINT RESULTS
print("-"*50)
print(f"EtE Mean Total Clicks Collected: {mean_EtE[-1]:.2f} ± {std_EtE:.2f}")
print(f"UCB Mean Total Clicks Collected: {mean_UCB[-1]:.2f} ± {std_UCB:.2f}")
print(f"MW Mean Total Clicks Collected:  {mean_MW[-1]:.2f} ± {std_MW:.2f}")
print(f"Optimal Expected Total Clicks:   {optimal[-1]:.2f}")

# CALCULATE OVERALL SELECTION PERCENTAGES
pct_arm0_EtE = np.mean(all_actions_EtE == 0) * 100
pct_arm1_EtE = np.mean(all_actions_EtE == 1) * 100

pct_arm0_UCB = np.mean(all_actions_UCB == 0) * 100
pct_arm1_UCB = np.mean(all_actions_UCB == 1) * 100

pct_arm0_MW = np.mean(all_actions_MW == 0) * 100
pct_arm1_MW = np.mean(all_actions_MW == 1) * 100

print("-" * 50)
print("ARM SELECTION PERCENTAGES (OVER ALL EPOCHS)")
print(f"EtE: Arm 0: {pct_arm0_EtE:>5.2f}%, Arm 1: {pct_arm1_EtE:>5.2f}%")
print(f"UCB: Arm 0: {pct_arm0_UCB:>5.2f}%, Arm 1: {pct_arm1_UCB:>5.2f}%")
print(f"MW:  Arm 0: {pct_arm0_MW:>5.2f}%, Arm 1: {pct_arm1_MW:>5.2f}%")
print("-" * 50)

"""
# Only for MW vs UCB
# Calculate absolute difference and % improvement FOR EACH EPOCH
diff_clicks_array = all_cum_reward_MW[:, -1] - all_cum_reward_UCB[:, -1]
pct_improvement_array = (diff_clicks_array / all_cum_reward_UCB[:, -1]) * 100

# Get the mean and standard deviation of those arrays
mean_diff_clicks = np.mean(diff_clicks_array)
std_diff_clicks = np.std(diff_clicks_array)

mean_pct_improvement = np.mean(pct_improvement_array)
std_pct_improvement = np.std(pct_improvement_array)

print("ALGORITHM COMPARISON (MW vs UCB)")
if mean_diff_clicks > 0:
    print(f"MW outperformed UCB by {mean_diff_clicks:.2f} ± {std_diff_clicks:.2f} clicks on average.")
    print(f"This is a {mean_pct_improvement:.2f}% ± {std_pct_improvement:.2f}% improvement over UCB.")
else:
    print(f"UCB outperformed MW by {abs(mean_diff_clicks):.2f} ± {std_diff_clicks:.2f} clicks on average.")
    print(f"MW performed {abs(mean_pct_improvement):.2f}% ± {std_pct_improvement:.2f}% worse than UCB.")
print("-" * 50)
"""

# PLOTS
# PLOT 1: Mean Cumulative Reward
plt.figure(figsize=(10,6))
plt.plot(mean_EtE, label="Explore-then-Exploit", linewidth=1)
plt.plot(mean_UCB, label="UCB", linewidth=1)
plt.plot(mean_MW, label="MW", linewidth=1)
plt.plot(optimal, '--', label="Optimal", linewidth=1)

plt.axvline(x=N*K, color='red', linestyle='--', label="EtE Exploit Start")

plt.title("Mean Cumulative Reward")
plt.xlabel("Round")
plt.ylabel("Average Total Clicks")

plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()
print()

# PLOTS 2 & 3: Rolling Selection Frequencies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

# --- Subplot 1: Arm 0 ---
ax1.plot(x_roll, mean_rolling_arm0_EtE, label="EtE", linewidth=1.5)
ax1.plot(x_roll, mean_rolling_arm0_UCB, label="UCB", linewidth=1.5)
ax1.plot(x_roll, mean_rolling_arm0_MW, label="MW", linewidth=1.5)
ax1.axvline(x=N * K, color='red', linestyle='--', label="EtE Exploit Start")
#ax1.axvline(x=T0_H, color='black', linestyle='--', label="Phase change") # Only for MW vs UCB

ax1.set_title(f"Rolling Selection Frequency of Arm 0 (window = {window})")
ax1.set_xlabel("Round")
ax1.set_ylabel("Frequency")
ax1.set_ylim(0, 1.05)
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

# --- Subplot 2: Arm 1 ---
ax2.plot(x_roll, mean_rolling_arm1_EtE, label="EtE", linewidth=1.5)
ax2.plot(x_roll, mean_rolling_arm1_UCB, label="UCB", linewidth=1.5)
ax2.plot(x_roll, mean_rolling_arm1_MW, label="MW", linewidth=1.5)
ax2.axvline(x=N * K, color='red', linestyle='--', label="EtE Exploit Start")
#ax2.axvline(x=T0_H, color='black', linestyle='--', label="Phase change") # Only for MW vs UCB

ax2.set_title(f"Rolling Selection Frequency of Arm 1 (window = {window})")
ax2.set_xlabel("Round")
ax2.set_ylabel("Frequency")
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()